# 面试题：工具 Schema 应怎样设计？

回答要点：schema 是可执行契约，至少定义输入类型、必填字段、范围/枚举、输出、读写副作用、权限、幂等语义、超时和版本。模型生成的参数未可信，服务端须校验并返回结构化错误。危险写操作需要确认令牌；JSON 格式合法不等于订单归属、额度和库存等业务前置条件已满足。

## 真实案例

退款工具处理六条请求，字段包括订单号、金额、原因和确认令牌。

## 基线

基线把模型给出的自由文本参数直接提交。

## 结果解读

手写 schema 输出每条失败的字段级错误，便于 Agent 定向修复。

## 失败案例

同一订单金额超出可退额度，即使 JSON 结构正确也必须拒绝。

In [1]:
requests = [{'id':'R01','order':'A12','amount':80,'reason':'破损','confirm':'yes'}, {'id':'R02','order':'A13','amount':0,'reason':'破损','confirm':'yes'}, {'id':'R03','order':'A14','amount':150,'reason':'破损','confirm':'yes'}, {'id':'R04','order':'A15','amount':30,'reason':'其他','confirm':'yes'}, {'id':'R05','order':'A16','amount':30,'reason':'错发','confirm':'no'}, {'id':'R06','order':'','amount':20,'reason':'破损','confirm':'yes'}]  # 构造六条退款参数，其中包含范围、枚举与确认错误。
refundable = {'A12':100,'A13':20,'A14':100,'A15':50,'A16':30}  # 构造权威订单可退额度的教学快照。
print('退款请求:', requests)  # 输出模型准备提交的原始参数。
print('权威可退额度:', refundable)  # 输出服务端用于业务校验的状态。

退款请求: [{'id': 'R01', 'order': 'A12', 'amount': 80, 'reason': '破损', 'confirm': 'yes'}, {'id': 'R02', 'order': 'A13', 'amount': 0, 'reason': '破损', 'confirm': 'yes'}, {'id': 'R03', 'order': 'A14', 'amount': 150, 'reason': '破损', 'confirm': 'yes'}, {'id': 'R04', 'order': 'A15', 'amount': 30, 'reason': '其他', 'confirm': 'yes'}, {'id': 'R05', 'order': 'A16', 'amount': 30, 'reason': '错发', 'confirm': 'no'}, {'id': 'R06', 'order': '', 'amount': 20, 'reason': '破损', 'confirm': 'yes'}]
权威可退额度: {'A12': 100, 'A13': 20, 'A14': 100, 'A15': 50, 'A16': 30}


In [2]:
baseline = [(row['id'], '提交') for row in requests]  # 构造自由文本参数一律提交的危险基线。
print('自由参数基线:', baseline)  # 输出基线忽略所有结构与业务约束的结果。
print('基线风险：零金额、未知订单、超额退款和未确认写操作都会穿透。')  # 解释不做 schema 门禁的具体风险。

自由参数基线: [('R01', '提交'), ('R02', '提交'), ('R03', '提交'), ('R04', '提交'), ('R05', '提交'), ('R06', '提交')]
基线风险：零金额、未知订单、超额退款和未确认写操作都会穿透。


In [3]:
def validate_refund(row):  # 定义退款工具在服务端执行的手写 schema 与业务校验。
    errors = []  # 初始化字段级错误列表。
    if row['order'] not in refundable:  # 检查订单标识是否存在于权威状态。
        errors.append('order:未知订单')  # 记录不可执行的订单错误。
    if not isinstance(row['amount'], int) or row['amount'] <= 0:  # 检查金额的类型与正数范围。
        errors.append('amount:必须为正整数')  # 返回机器可读的金额错误。
    if row['reason'] not in {'破损','错发','未收到'}:  # 检查原因是否属于稳定枚举。
        errors.append('reason:不在枚举')  # 防止自由文本绕开审计分类。
    if row['confirm'] != 'yes':  # 检查写操作是否有显式确认令牌。
        errors.append('confirm:需要确认')  # 阻断模型猜测式退款。
    if row['order'] in refundable and row['amount'] > refundable[row['order']]:  # 检查跨字段的额度前置条件。
        errors.append('amount:超过可退额度')  # 返回需要改金额或升级的业务错误。
    return errors  # 将全部错误返回给 Agent 进行有限轮修复。

In [4]:
results = [(row['id'], validate_refund(row)) for row in requests]  # 对六条请求运行服务端校验。
print('id | 校验结果')  # 输出字段级校验结果表标题。
for request_id, errors in results:  # 遍历每条退款请求的门禁结果。
    print(request_id, '通过' if not errors else errors)  # 输出通过或全部结构化错误。
print('可安全提交数:', sum(not errors for _, errors in results))  # 汇总真正通过全部约束的请求数量。

id | 校验结果
R01 通过
R02 ['amount:必须为正整数']
R03 ['amount:超过可退额度']
R04 ['reason:不在枚举']
R05 ['confirm:需要确认']
R06 ['order:未知订单']
可安全提交数: 1


In [5]:
json_like_ok = requests[2]['order'] != '' and isinstance(requests[2]['amount'], int)  # 模拟只验证 JSON 外形的错误检查。
full_errors = validate_refund(requests[2])  # 使用 schema 加权威额度检查同一条超额请求。
print('失败案例 R03：外形有效=', json_like_ok, '，完整校验=', full_errors)  # 展示 JSON 合法不等于业务合法。
print('生产差距：生产 schema 要版本化、由服务端强制、记录审计字段，并把库存、归属和审批做成不可绕过的领域校验。')  # 说明教学门禁与真实服务接口差异。

失败案例 R03：外形有效= True ，完整校验= ['amount:超过可退额度']
生产差距：生产 schema 要版本化、由服务端强制、记录审计字段，并把库存、归属和审批做成不可绕过的领域校验。


In [6]:
assert validate_refund(requests[0]) == []  # 验证完整合法的退款参数能通过。
assert 'amount:超过可退额度' in full_errors  # 验证超额金额会触发业务前置条件。
assert validate_refund(requests[4]) == ['confirm:需要确认']  # 验证未确认写操作会被阻断。